In [29]:
from datetime import datetime, timedelta
import calendar
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr

In [ ]:
init_date = datetime(2023, 10, 1, 0)
end_date = datetime(2026, 8, 31, 23)

date_list = []
current_date = init_date

while current_date <= end_date:
    date_list.append(current_date)
    current_date += timedelta(hours=1)


In [12]:
hours = [date.hour for date in date_list]
week_days = [date.weekday() for date in date_list]
month_days = [date.day for date in date_list]
months = [date.month for date in date_list]
# year = [date.year for date in date_list]  ## Necessário?

In [ ]:

def get_sines_and_cosines(original_date, values, type):

    if type == "hour":
        fase = [value/24 for value in values]
    elif type == "week_day":
        fase = [value/7 for value in values]
    elif type == "month_day":
        max_month_days = [calendar.monthrange(date.year, date.month)[1] for date in original_date]
        fase = [(value-1)/max_day for value, max_day in zip(values, max_month_days)]
    elif type == "month":
        fase = [value/12 for value in values]
    
    sines = [ np.sin(2*np.pi*f) for f in fase]
    cosines = [np.cos(2*np.pi*f) for f in fase]

    return sines, cosines


hour_sines, hour_cosines = get_sines_and_cosines(date_list, hours, "hour")
week_day_sines, week_day_cosines = get_sines_and_cosines(date_list, week_days, "week_day")
month_day_sines, month_day_cosines = get_sines_and_cosines(date_list, month_days, "month_day")
month_sines, month_cosines = get_sines_and_cosines(date_list, months, "month")


In [32]:
date_features = xr.Dataset(
    data_vars={
        "hour_sin": ("time", hour_sines),
        "hour_cos": ("time", hour_cosines),
        "week_day_sin": ("time", week_day_sines),
        "week_day_cos": ("time", week_day_cosines),
        "month_day_sin": ("time", month_day_sines),
        "month_day_cos": ("time", month_day_cosines),
        "month_sin": ("time", month_sines),
        "month_cos": ("time", month_cosines),
    },
    coords={"time": date_list},
)

date_features.to_netcdf("../data/date_features.nc", )